In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 38. Week 26 — VAR, predictive content, impulse responses, and cointegration

## 学習目標

- fixed-decay NS factorへVARをfitできる
- Granger predictive contentとstructural causalityを区別できる
- reduced-form IRFとorthogonalized IRFのordering依存を説明できる
- cointegrationを高いlevel correlationと区別できる

## 前提知識

- Week 25のstationarityとAR
- B1のNelson–Siegel loading

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 38


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. VAR and information set

$$
x_t=c+A_1x_{t-1}+\cdots+A_px_{t-p}+u_t.
$$

ここで (x_t) はlevel、slope、curvature factor。Granger検定の帰無仮説は「指定したlagと線形情報集合の中で追加の予測力がない」であり、政策的・構造的因果ではない。

In [4]:
decay = 0.5
factors = qt.extract_nelson_siegel_factors(curve_yields, maturity_years, decay)
factor_names = ["level", "slope", "curvature"]
factor_model = qt.fit_var(factors[train_mask], 1)
granger_rows = []
for effect_index, effect_name in enumerate(factor_names):
    for cause_index, cause_name in enumerate(factor_names):
        if effect_index != cause_index:
            result = qt.granger_causality_test(
                np.diff(factors[train_mask, effect_index]),
                np.diff(factors[train_mask, cause_index]),
                lags=1,
            )
            granger_rows.append(
                {"effect": effect_name, "cause": cause_name, "f_statistic": result.f_statistic, "p_value": result.p_value}
            )
display(pd.DataFrame(granger_rows))

,effect,cause,f_statistic,p_value
0,level,slope,1.685830,0.194333
1,level,curvature,0.303375,0.581849
2,slope,level,1.204463,0.272592
3,slope,curvature,0.820558,0.365149
4,curvature,level,2.188598,0.139227
5,curvature,slope,1.708912,0.191309


In [5]:
responses = qt.impulse_response(factor_model, 20, orthogonalized=False)
fig = go.Figure()
for target_index, target_name in enumerate(factor_names):
    fig.add_scatter(
        x=np.arange(21),
        y=responses[:, target_index, 0],
        name=f"level shock to {target_name}",
        mode="lines+markers",
    )
fig.update_layout(
    title="Reduced-form VAR response to a unit level-factor innovation",
    xaxis_title="Publication horizon",
    yaxis_title="Factor response",
    template="plotly_white",
)
fig.show()

## 2. Cointegration diagnostic

二つのlevel系列が非定常でも、ある (eta) について (y_t-\beta x_t) が定常ならcointegratedである。次のtwo-step diagnosticは正式なEngle–Granger critical valueを実装していないため、ordinary p-valueを出さない。

In [6]:
level_design = np.column_stack([np.ones(np.sum(train_mask)), curve_yields[train_mask, 1]])
cointegration_beta = np.linalg.lstsq(level_design, curve_yields[train_mask, 3], rcond=None)[0]
cointegration_residual = curve_yields[train_mask, 3] - level_design @ cointegration_beta
cointegration_diagnostic = qt.dickey_fuller_diagnostic(cointegration_residual)
display(
    pd.DataFrame(
        [
            {
                "relationship": "10y on 2y",
                "intercept": cointegration_beta[0],
                "slope": cointegration_beta[1],
                "residual_df_t": cointegration_diagnostic.t_statistic,
                "calibrated_p_value_available": False,
            }
        ]
    )
)

,relationship,intercept,slope,residual_df_t,calibrated_p_value_available
0,10y on 2y,1.200218,0.656485,-2.013846,False


## 3. Validation forecast

In [7]:
horizon = 5
origins = np.flatnonzero(
    (curve_dates > train_end_date)
    & (curve_dates <= validation_end_date)
    & (np.arange(curve_dates.size) + horizon < curve_dates.size)
)
origins = origins[curve_dates[origins + horizon] <= validation_end_date]
var_curve_predictions = np.vstack(
    [qt.forecast_var(factor_model, factors[: origin + 1], horizon)[-1] @ qt.nelson_siegel_loadings(maturity_years, decay).T for origin in origins]
)
random_walk_predictions = curve_yields[origins]
actual = curve_yields[origins + horizon]
display(
    pd.DataFrame(
        {
            "model": ["random walk", "factor VAR(1)"],
            "aggregate_rmse_bp": [
                100.0 * np.sqrt(np.mean((actual - random_walk_predictions) ** 2)),
                100.0 * np.sqrt(np.mean((actual - var_curve_predictions) ** 2)),
            ],
        }
    )
)

,model,aggregate_rmse_bp
0,random walk,15.182518
1,factor VAR(1),18.642397


## 4. 失敗モード

- Granger predictive contentを因果効果と呼ぶ
- level VARの高いfitだけを報告する
- Cholesky orderingを隠してorthogonalized IRFを構造shockと呼ぶ
- correlationだけでcointegrationと結論する
- factor extractionのdecayをouter testで選ぶ

## 5. 段階別演習

### 基礎

1. companion formでVAR(1)の安定性条件を書け。
2. reduced-formとorthogonalized IRFを比較せよ。

### 標準

3. factor orderingを変えてCholesky IRFの感応度を測れ。
4. VAR(1)とseparate AR(1)をvalidationで比較せよ。

### 研究

5. local projectionのestimandとHAC inference contractを設計せよ。

## 6. Exit Criteria

- [ ] VARへstationary transformationを検討した
- [ ] Grangerとcausalityを分離した
- [ ] IRFのshock normalizationを記録した
- [ ] cointegrationをresidual stationarityで定義した
- [ ] validation forecastをrandom walkと比較した

## 7. 出典


- [Forecasting: Principles and Practice — Stationarity and differencing](https://otexts.com/fpp3/stationarity.html)
- [Forecasting: Principles and Practice — ARIMA models](https://otexts.com/fpp3/arima.html)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)

- [Sims (1980), Macroeconomics and Reality](https://doi.org/10.2307/1912017)